# 05 · Putting it all together: deploy to the Arduino

**Goals of this notebook**

1. Combine **distillation → node pruning → quantization** into one pipeline.
2. Meet the challenge from notebook 01: **≤ 10 KB** of parameters, at most **1 point** less accurate than the baseline.
3. Turn the model into C++ code (`model_data.h`) and check that the C++ code computes *exactly* what Python computes.
4. Upload it to the Nano 33 BLE Sense and recognise gestures in real time.
5. Measure the real cost on the board, and (optionally) retrain on your own recordings.

### Settings
Change these two lines if needed, then run the whole notebook (*Run → Run All Cells*). The training takes about a minute.

In [ ]:
DATA_SOURCE = "synthetic"     # "synthetic", or "recorded" to use your own data/recorded/*.csv (notebook 01, section 8)
FINAL_HIDDEN = (24, 12)       # size of the deployed network after node pruning

In [ ]:
import copy

import numpy as np
import torch
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# EXTERNAL helpers (imported from the course folder `aiedge`). Each one is written out in
# full in an earlier notebook; this notebook only combines them into one pipeline.
#
#   load_gesture_data            synthetic sensor data (comment block in notebook 01)
#   MLP, count_params, mcu_cost  the small network and its cost counters (notebook 01)
#   train, accuracy, set_seed,   training loop, test accuracy, repeatable randomness,
#   confusion_matrix             and the table of right/wrong answers (notebook 01)
#   ConvNet                      the teacher network (notebook 04)
#   prune_neurons                node pruning (notebook 03)
#   QuantMLP, int8_forward,      quantization-aware training and the int8 inference
#   int8_accuracy,               engine (notebook 02)
#   int8_model_bytes
# Why external: they are long, and copying them here would mean the same code lives in
# several notebooks. The two board-specific helpers further down (C header export and
# serial port reader) have their own comment blocks.
# ---------------------------------------------------------------------------
import sys, pathlib
# Only needed so Python can find the course folder `aiedge` (the helpers imported below).
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from aiedge.data import load_gesture_data
from aiedge.models import MLP, ConvNet, count_params
from aiedge.training import train, accuracy, set_seed, confusion_matrix
from aiedge.pruning import prune_neurons
from aiedge.quantization import QuantMLP, int8_forward, int8_accuracy, int8_model_bytes
from aiedge.export import mcu_cost

GESTURES = ("punch", "lift", "wave", "circle", "twist")   # the 5 classes

data = load_gesture_data(source=DATA_SOURCE)
print(data.summary())

## 1. The plan, and why in this order

```
 teacher (ConvNet)
        │ knowledge distillation
        ▼
 student MLP 300-64-32-5 (float32)  ──node pruning + distilled fine-tuning──►  300-24-12-5 (float32)
                                                                                      │ quantization-aware training (int8)
                                                                                      ▼
                                                                         300-24-12-5 (int8)  ──►  model_data.h  ──►  Arduino
```

* **Distillation first**: it gives the best possible starting point, a student that already carries the teacher's knowledge. We keep using the teacher during all later fine-tuning steps too.
* **Pruning second**: it changes the *structure* of the network. Every structural change needs fine-tuning, so it should happen before the final step.
* **Quantization last**: the scales and zero-points depend on the exact final weights. Any change after quantization would invalidate them. QAT is the final fine-tuning, done at the precision the board will use.

### Step 0: the reference points
First, the baseline from notebook 01, the model to measure the challenge against. One training run can be lucky or unlucky by about a point (notebook 01), and a challenge should not be decided by luck. So we train the baseline with **three seeds and use the average** as the reference.

Then the teacher from notebook 04. If you ran notebook 04 on the same data, its teacher is loaded from disk; otherwise it is trained here.

In [ ]:
base_runs = []
for s in [0, 1, 2]:
    set_seed(s)
    baseline = MLP(hidden=(64, 32))
    train(baseline, data, augment=True, seed=s, verbose=False)
    base_runs.append(accuracy(baseline, data.X_test, data.y_test))
base_acc = float(np.mean(base_runs))
print("baseline runs:", [f"{a:.1%}" for a in base_runs], f"-> reference {base_acc:.1%}")

teacher = ConvNet()
teacher_file = ROOT / "models" / "teacher.pt"
if DATA_SOURCE == "synthetic" and teacher_file.exists():
    teacher.load_state_dict(torch.load(teacher_file))
    print("teacher loaded from", teacher_file)
else:
    set_seed(0)
    train(teacher, data, augment=True, verbose=False)
teacher.eval()
print(f"teacher: {accuracy(teacher, data.X_test, data.y_test):.1%}")

### Step 1: distil into a 64-32 student

In [ ]:
KD = dict(teacher=teacher, temperature=4.0, alpha=0.5, augment=True, verbose=False)

set_seed(0)
student = MLP(hidden=(64, 32))
train(student, data, **KD)
print(f"distilled student {student}: {accuracy(student, data.X_test, data.y_test):.1%}")

### Step 2: node pruning to the final size, fine-tuned with the teacher

In [ ]:
pruned = prune_neurons(student, data.X_train, FINAL_HIDDEN)
print(f"pruned to {pruned}, before fine-tuning: {accuracy(pruned, data.X_test, data.y_test):.1%}")
train(pruned, data, lr=1e-3, **KD)
print(f"after fine-tuning:                    {accuracy(pruned, data.X_test, data.y_test):.1%}")

### Step 3: int8 quantization-aware training
We calibrate the activation ranges, then fine-tune with fake quantization (and still with the teacher) for a few epochs at a small learning rate.

In [ ]:
qmodel = QuantMLP(copy.deepcopy(pruned))
qmodel.calibrate(data.X_train)
print(f"int8 after calibration (PTQ): {accuracy(qmodel, data.X_test, data.y_test):.1%}")
train(qmodel, data, lr=5e-4, epochs=40, patience=10, **KD)

int8_model = qmodel.to_int8()
final_acc = int8_accuracy(int8_model, data.X_test, data.y_test)
final_bytes = int8_model_bytes(int8_model)
print(f"int8 after QAT, int8 inference engine: {final_acc:.1%}")

## 2. Did we meet the challenge?

In [ ]:
def cost_table(models):
    """Print-ready table. `models` maps a name to (model, is_int8, test_accuracy).
    Parameters, MACs, flash and estimated time come from `mcu_cost` (notebook 01)."""
    head = f"{'model':<34}{'acc':>7}{'params':>9}{'MACs':>9}{'flash KB':>10}{'est. ms':>9}"
    lines = [head, "-" * len(head)]
    for name, (m, int8, acc) in models.items():
        cost = mcu_cost(m, int8)
        lines.append(f"{name:<34}{acc:>7.3f}{cost['params']:>9,}{cost['MACs']:>9,}"
                     f"{cost['flash_KB']:>10.1f}{cost['est_ms']:>9.2f}")
    return "\n".join(lines)


print(cost_table({
    "baseline 64-32 (float32, 3-seed mean)": (baseline, False, base_acc),
    "distilled 64-32 (float32)": (student, False, accuracy(student, data.X_test, data.y_test)),
    f"pruned {FINAL_HIDDEN[0]}-{FINAL_HIDDEN[1]} (float32)": (pruned, False, accuracy(pruned, data.X_test, data.y_test)),
    f"FINAL {FINAL_HIDDEN[0]}-{FINAL_HIDDEN[1]} (int8)": (qmodel.mlp, True, final_acc),
}))
base_bytes = 4 * count_params(baseline)
print(f"\nmodel memory: {base_bytes / 1024:.1f} KB -> {final_bytes / 1024:.1f} KB "
      f"({base_bytes / final_bytes:.1f}x smaller)")
print(f"accuracy:     {base_acc:.1%} -> {final_acc:.1%} ({100 * (final_acc - base_acc):+.1f} points)")
ok = final_bytes <= 10 * 1024 and final_acc >= base_acc - 0.01
print("\n🎯 challenge (≤ 10 KB, ≥ baseline - 1 point):", "MET ✅" if ok else "not met yet ❌, try another FINAL_HIDDEN")

In [ ]:
y_pred = int8_forward(int8_model, data.X_test.numpy()).argmax(1)
cm = confusion_matrix(data.y_test.numpy(), y_pred)

fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cm, cmap="Blues")
ax.grid(False)
ax.set_xticks(range(len(GESTURES)), GESTURES, rotation=45)
ax.set_yticks(range(len(GESTURES)), GESTURES)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title(f"Final int8 model: {final_acc:.1%}")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):     # write the count into every square
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)
plt.show()

What did each technique contribute?
* **Node pruning** cut the parameters and MACs by ~3x, in a form our plain C code profits from directly.
* **Quantization** cut the bytes by another ~4x, essentially for free in accuracy.
* **Distillation** keeps the teacher's knowledge in the loop during every training step. With the large synthetic dataset its effect is small (notebook 04, experiment 1). With your own small recorded dataset (section 6) it can add a few points (notebook 04, experiment 2).

(The exact numbers vary by about ±1 point between runs, see notebook 01. If the challenge is narrowly missed, re-run or try `FINAL_HIDDEN = (32, 16)`; note that this is slightly above 10 KB.)

## 3. From PyTorch to C++

The Arduino has no Python and no PyTorch. Everything the board needs is written into a C++ header file, **`arduino/gesture_classifier/model_data.h`**:
* the settings (sampling rate, window length, trigger thresholds, gesture names);
* the normalisation constants (mean and std per channel, from the *training* set);
* the int8 weights, int32 biases, multipliers and zero-points of every layer;
* the float32 weights of the same network (to compare on the board);
* one test gesture with the logits Python computed for it, for the **self-test**.

The arrays are declared `const`, so the compiler puts them in **flash**, not RAM. The inference code itself is in `edge_nn.h`: about 100 lines of plain C++ that implement the formulas from notebook 02. Open both files and have a look!

In [ ]:
# ---------------------------------------------------------------------------
# EXTERNAL helper: write_model_header (aiedge/export.py)
#   What it does: takes the trained int8 network (`qmodel`) and the data (for the
#                 normalisation constants) and writes everything the board needs as C++
#                 arrays into arduino/gesture_classifier/model_data.h. Returns the file path.
#   Why external: ~150 lines of string formatting for C++ that has nothing to do with
#                 machine learning. Read aiedge/export.py if you are curious.
# ---------------------------------------------------------------------------
from aiedge.export import write_model_header

header = write_model_header(qmodel, data)
print(f"written: {header}  ({header.stat().st_size / 1024:.0f} KB of text)")
print("".join(header.read_text().splitlines(keepends=True)[:12]))

### Check the C++ code on your laptop first
Debugging on a microcontroller is hard, so we first compile the *same* C++ inference code for your laptop (if the `g++` compiler is installed) and run it on all test gestures. The int8 results must be **bit-identical** to Python's `int8_forward`. Not just close: identical, because both do exactly the same integer operations and the same float32 roundings.

In [ ]:
# ---------------------------------------------------------------------------
# EXTERNAL helper: check_with_host_compiler (aiedge/export.py)
#   What it does: compiles arduino/gesture_classifier/edge_nn.h with the laptop's g++,
#                 runs it on every test gesture and compares with Python's int8_forward.
#                 Prints a message instead if g++ is not installed.
#   Why external: it calls a compiler and runs a program, which is system plumbing.
# ---------------------------------------------------------------------------
from aiedge.export import check_with_host_compiler

check_with_host_compiler(qmodel, data, header)

## 4. Upload to the board

1. Open **`arduino/gesture_classifier/gesture_classifier.ino`** in the Arduino IDE. (The folder also contains `edge_nn.h`, `gesture_recorder.h` and the freshly generated `model_data.h`; the IDE shows them as tabs.)
2. At the top of the sketch, set **`#define BOARD_REVISION`** to `1` or `2` (see notebook 00). Leave `MODEL_PRECISION` at `8`.
3. Select the board *Arduino Nano 33 BLE* and its port, and click **Upload**. The IDE prints how much flash ("program storage") and RAM ("dynamic memory") the sketch uses. **Write these numbers down.**
4. Open the **Serial Monitor** at **115200 baud**. You should see something like:

```
# gesture_classifier: int8 network, 3 layers, input 300, gestures: punch lift wave circle twist
# self-test logits (board):  -3.1416 7.2734 ...
# self-test logits (Python): -3.1416 7.2734 ...
# self-test: true gesture = lift, predicted = lift, max difference = 0.000000  -> PASS
# inference time: 450.3 us per gesture
# ready -- make a gesture!
```

5. Hold the board as described in notebook 00 and make gestures! Each line shows the prediction, the probabilities and the inference time, and the RGB LED lights up: **punch = red, lift = green, wave = blue, circle = yellow, twist = purple**, off = "not sure".

**Command line alternative** (`arduino-cli`, use `Arduino_BMI270_BMM150` for Rev2, and your own port):
```bash
arduino-cli core install arduino:mbed_nano
arduino-cli lib install Arduino_LSM9DS1
arduino-cli compile -b arduino:mbed_nano:nano33ble arduino/gesture_classifier
arduino-cli upload  -b arduino:mbed_nano:nano33ble -p /dev/ttyACM0 arduino/gesture_classifier
arduino-cli monitor -p /dev/ttyACM0 -c baudrate=115200
```

Instead of the Serial Monitor you can also watch the predictions from here (close the Serial Monitor first):

In [ ]:
WATCH = False      # set to True while the board runs gesture_classifier
if WATCH:
    # EXTERNAL helper: show_predictions (aiedge/board.py)
    #   What it does: opens the board's serial port and prints each prediction line for
    #                 `seconds` seconds. Needs the `pyserial` package and a connected board.
    #   Why external: serial port code is plumbing, not machine learning.
    from aiedge.board import show_predictions
    show_predictions(seconds=60)          # add port="..." if the board is not found

### Troubleshooting

| Problem | Likely cause / fix |
|---|---|
| `IMU not found. Is BOARD_REVISION set correctly?` | Wrong revision: change `BOARD_REVISION` and re-upload. |
| Compile error `Arduino_LSM9DS1.h: No such file` (or BMI270) | Install the library for your revision (notebook 00, step 3). |
| Upload fails / port not found | Double-press the reset button (orange LED "breathes"), select the port again, upload. |
| Self-test **FAIL** | `model_data.h` in the sketch folder is not the one you just generated, or you edited `edge_nn.h`. Re-run this notebook. |
| Nothing happens when moving | Move a bit more briskly; the trigger needs 0.35 g or 120 °/s. The orange LED is on while recording. |
| Gestures are recognised poorly | The synthetic data may differ from your board / your style. Record your own data (section 6). |
| Garbage characters | Set the Serial Monitor to 115200 baud. |

## 5. Measure the real cost on the board ⭐

The numbers so far were *estimates*. Now measure. Upload the sketch twice: once with `#define MODEL_PRECISION 8` and once with `32`. Fill in this table (the "sketch" numbers come from the IDE's compile output, the time from the self-test):

| | int8 | float32 |
|---|---|---|
| sketch flash ("program storage") | | |
| sketch RAM ("dynamic memory") | | |
| inference time (µs) | | |
| estimated time from the cost model (µs) | see below | see below |

In [ ]:
c = mcu_cost(qmodel.mlp)
print(f"MACs: {c['MACs']:,}  ->  estimated {1000 * c['est_ms']:.0f} us at ~4 cycles/MAC")
print(f"weight bytes: int8 {final_bytes:,} vs float32 {4 * count_params(qmodel.mlp):,}  "
      f"-> expected flash difference ~{(4 * count_params(qmodel.mlp) - final_bytes) / 1024:.0f} KB")

Questions to discuss:
* Is the **flash** difference between the int8 and float32 sketches what you expected? Note that most of the flash is used by the Arduino core itself (~95 KB), not by the model.
* Is int8 **faster** than float32 on this chip? Remember the Cortex-M4F has a hardware FPU, and our C loop does one multiplication at a time. Optimised libraries (CMSIS-NN, used by TensorFlow Lite Micro) use special instructions that multiply two 16-bit values per cycle (`SMLAD`), which is where int8 gets its speed advantage.
* How many cycles per MAC did you measure: (time in µs × 64) / MACs? How good was our "4 cycles" guess?
* The board needs 1 second to record a gesture and well under 1 ms to classify it. What does that mean for the energy budget: where would you save energy first?

## 6. Train on your own data ⭐

The synthetic data is an approximation. For the best results on your board:
1. Record ~40 examples of every gesture with `gesture_capture` (notebook 01, section 8).
2. Set `DATA_SOURCE = "recorded"` at the top of this notebook and **run all cells** again.
3. Upload the sketch again (the new `model_data.h` is in place) and test.

You could also add a sixth gesture of your own: add its name to `GESTURES` in `aiedge/config.py`, record it, retrain. (With synthetic data that's not possible: the simulator only knows five gestures.)

## 7. Where to go from here

You have now done what professional edge-AI engineers do, from scratch, without hiding anything inside a framework. In practice you would use tools that automate parts of this:
* **TensorFlow Lite for Microcontrollers / LiteRT** and **CMSIS-NN**: int8 inference engines with optimised kernels for Arm Cortex-M.
* **Edge Impulse**: a web platform for collecting data, training and deploying to boards like this one.
* **PyTorch's `torchao`** and **ExecuTorch**: quantization tools and an on-device runtime.

Further reading:
* Han, Mao & Dally (2016), *Deep Compression: pruning, trained quantization and Huffman coding*.
* Jacob et al. (2018), *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*. This is the int8 scheme you implemented.
* Hinton, Vinyals & Dean (2015), *Distilling the Knowledge in a Neural Network*.
* Frankle & Carbin (2019), *The Lottery Ticket Hypothesis*.
* Warden & Situnayake (2019), *TinyML* (O'Reilly), a book about exactly this kind of project.

## Final exercises

1. ⭐ **Fill in the table in section 5** and answer the questions below it.
2. ⭐ **Push the challenge.** What is the smallest `FINAL_HIDDEN` that still meets the "-1 point" rule? Can you get below 5 KB?
3. **Confidence threshold.** Change `CONFIDENCE_THRESHOLD` in the sketch (0.6 by default). What happens with random movements (not one of the 5 gestures) at 0.4 and at 0.9?
4. **Without distillation.** Replace `**KD` by `augment=True, verbose=False` everywhere (no teacher). How much accuracy does the final int8 model lose with the synthetic data? And with your own recorded data?
5. **Ablation.** Which of the three techniques contributes most to the size reduction, and which costs the most accuracy? Make a table, leaving out one technique at a time.
6. **Challenge.** The model only looks at one window after a trigger. How would you recognise gestures *continuously*, without a trigger? What would change in the data, the model and the sketch?